# 33 — Prior-method code alignment audit

This notebook stops the architecture-rotation cycle and audits what was actually implemented. It compares repository code with primary papers and official implementations, then states exactly which previous negative results are trustworthy and which only reject a project-specific approximation. No model is trained, and no RevalExo or NONAN signal is loaded.

## Project scenario used for fit assessment

- Binary participant-level stroke-versus-healthy classification.
- Primary input: one lower-back acceleration-magnitude channel, 500 samples at 100 Hz.
- Development domains: Felius, Voisard, and Sint; complete-source holdout is required for transport claims.
- RevalExo and NONAN are frozen and cannot select architecture, normalization, calibration, threshold, or augmentation.
- Synthetic windows may augment training but never count as independent participants.

## Upstream code inspected

The audit used shallow clones of the official repositories and recorded their exact commits: [InceptionTime](https://github.com/hfawaz/InceptionTime), [MiniROCKET](https://github.com/angus924/minirocket), [HAROOD](https://github.com/AIFrontierLab/HAROOD), [ERM++](https://github.com/piotr-teterwak/erm_plusplus), [AttentionDeepMIL](https://github.com/AMLab-Amsterdam/AttentionDeepMIL), [group_DRO](https://github.com/kohpangwei/group_DRO), [HAR-Bench](https://github.com/saiketa/HAR-Bench), and [IMUEval](https://github.com/H-IAAC/synth-imu-eval). IMUDiffusion has a detailed primary paper but no discoverable official repository; the comparison therefore uses its published architecture and training specification rather than third-party code.


In [1]:
from pathlib import Path
import hashlib, json, sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
PROCESSED = ROOT / 'data' / 'processed'

local_files = {
    'compact_inception': ROOT / 'models/stroke_gait_inception.py',
    'domain_generalization': ROOT / 'src/models/evidence_gated_domain_generalization.py',
    'group_dro': ROOT / 'scripts/benchmark_group_robust_leave_one_source_out.py',
    'minirocket_rescue': ROOT / 'src/models/lower_back_minirocket_rescue.py',
    'participant_mil': ROOT / 'src/models/participant_attention_mil.py',
    'ddpm_v2': ROOT / 'scripts/train_heldout_conditional_healthy_diffusion.py',
    'physical_augmentation': ROOT / 'scripts/benchmark_physical_augmentation_voisard.py',
    'normalization': ROOT / 'scripts/benchmark_normalization_variants.py',
    'hard_negative': ROOT / 'scripts/benchmark_hard_negative_exposure_primary_oof.py',
}
assert all(path.exists() for path in local_files.values())
text = {name: path.read_text(encoding='utf-8').lower() for name, path in local_files.items()}
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
inventory = pd.DataFrame([{'component': name, 'path': str(path.relative_to(ROOT)), 'sha256': sha256(path), 'lines': len(text[name].splitlines())} for name, path in local_files.items()])
display(inventory)
print('Frozen external arrays loaded by this notebook: NONE')


,component,path,sha256,lines
0,compact_inception,models\stroke_gait_inception.py,91ad854344972dd45b3d7c236e26b47be07f44b7c1c065...,55
1,domain_generalization,src\models\evidence_gated_domain_generalizatio...,c3ee3e350e48ce578f386f8a04cecb6bc156ecd78470ab...,690
2,group_dro,scripts\benchmark_group_robust_leave_one_sourc...,bb752d7a9cd769e94bda369a202bc62a3574dbe65389c1...,38
3,minirocket_rescue,src\models\lower_back_minirocket_rescue.py,a54333326767f22e62fea0043f95120ed2ee5ff04ef480...,309
4,participant_mil,src\models\participant_attention_mil.py,21425efa8dacbe3ab2cb5f924fe6ca69865f8605e7cee1...,506
5,ddpm_v2,scripts\train_heldout_conditional_healthy_diff...,4da3b704ef129e84988af17d219761be7be9d3fa3c68ac...,242
6,physical_augmentation,scripts\benchmark_physical_augmentation_voisar...,e39b0e70c1f31e33f133a653f3aac9bc8518573140899f...,30
7,normalization,scripts\benchmark_normalization_variants.py,03b2dc5eed977e015daee6b94407725f692c29a7f930e4...,46
8,hard_negative,scripts\benchmark_hard_negative_exposure_prima...,212672ba6cb4771f8afbde825a6c34b4a8aebabe658e91...,86


Frozen external arrays loaded by this notebook: NONE


In [2]:
checks = [
    ('Compact CNN', 'two local Inception blocks', text['compact_inception'].count('inceptionblock(') - 1 == 2, 'Official InceptionTime uses depth 6'),
    ('Compact CNN', 'local kernels 7/15/25', all(k in text['compact_inception'] for k in ('7, padding=3', '15, padding=7', '25, padding=12')), 'Official kernels are approximately 40/20/10'),
    ('CORAL', 'mean and covariance alignment', all(k in text['domain_generalization'] for k in ('mean_first', 'covariance_first', 'coral_weight')), 'Matches HAROOD core objective'),
    ('ERM++-style', 'head warm-up and SMA present', all(k in text['domain_generalization'] for k in ('linear_steps', 'simplemovingaverage', 'sma_start_step')), 'Only a subset of ERM++'),
    ('ERM++-style', 'pretrained initialization absent', 'pretrained' not in text['domain_generalization'], 'Official ERM++ relies on pretrained backbones and full-data retraining'),
    ('GroupDRO', 'exponential q update present', 'q=q*torch.exp(.1*gl.detach())' in text['group_dro'].replace(' ',''), 'Core update is present'),
    ('GroupDRO', 'groups are source by class', 'for cls in [0,1]' in text['group_dro'], 'HAROOD q indexes source-domain minibatches'),
    ('MiniROCKET rescue', '2,000 features / 16 dilations', all(k in text['minirocket_rescue'] for k in ('num_kernels=2000', 'max_dilations_per_kernel=16')), 'Canonical defaults are 10,000 / 32'),
    ('MiniROCKET rescue', 'logistic head used', 'logisticregression' in text['minirocket_rescue'], 'Canonical pipeline uses scaled RidgeClassifierCV'),
    ('Attention MIL', 'gated attention operator', all(k in text['participant_mil'] for k in ('torch.tanh', 'torch.sigmoid', 'torch.softmax')), 'Matches Ilse gated-attention operator'),
    ('Attention MIL', 'one bag-level BCE loss', 'binary_cross_entropy_with_logits' in text['participant_mil'], 'Matches bag-level Bernoulli objective'),
    ('DDPM v2', '100-step time-domain generator', all(k in text['ddpm_v2'] for k in ('timesteps = int', '100', 'conv1d')), 'Not the 3,000-step STFT IMUDiffusion model'),
    ('DDPM v2', 'no STFT or multihead attention', 'stft' not in text['ddpm_v2'] and 'multiheadattention' not in text['ddpm_v2'], 'Material architecture mismatch'),
    ('Physical augmentation', 'custom gain/noise/time scaling', all(k in text['physical_augmentation'] for k in ('gain=', 'normal(0,.01', 'scale=rng.uniform')), 'Not PPDA/WIMUSim physics simulation'),
    ('Normalization', 'RevalExo loaded in comparison script', 'revalexo_external_windows_float32.npy' in text['normalization'], 'External results from this script cannot select normalization'),
]
check_table = pd.DataFrame(checks, columns=['family', 'local_fact', 'verified', 'upstream_implication'])
assert check_table.verified.all()
display(check_table)


,family,local_fact,verified,upstream_implication
0,Compact CNN,two local Inception blocks,True,Official InceptionTime uses depth 6
1,Compact CNN,local kernels 7/15/25,True,Official kernels are approximately 40/20/10
2,CORAL,mean and covariance alignment,True,Matches HAROOD core objective
3,ERM++-style,head warm-up and SMA present,True,Only a subset of ERM++
4,ERM++-style,pretrained initialization absent,True,Official ERM++ relies on pretrained backbones ...
5,GroupDRO,exponential q update present,True,Core update is present
6,GroupDRO,groups are source by class,True,HAROOD q indexes source-domain minibatches
7,MiniROCKET rescue,"2,000 features / 16 dilations",True,"Canonical defaults are 10,000 / 32"
8,MiniROCKET rescue,logistic head used,True,Canonical pipeline uses scaled RidgeClassifierCV
9,Attention MIL,gated attention operator,True,Matches Ilse gated-attention operator


In [3]:
upstream = pd.DataFrame([
    ('InceptionTime', 'https://github.com/hfawaz/InceptionTime', '952d115e83fb3a66d75c5858c702f8dd5eeb18b7'),
    ('MiniROCKET', 'https://github.com/angus924/minirocket', '0b1c245d9c9dbc50886f28bc7b32d5d45b5663d6'),
    ('HAROOD', 'https://github.com/AIFrontierLab/HAROOD', '3c2ce00e2b408ddd913c3d179cd86daf0c44bc90'),
    ('ERM++', 'https://github.com/piotr-teterwak/erm_plusplus', '2535f5d21d16e4fadcec79909835fac97cff6d63'),
    ('AttentionDeepMIL', 'https://github.com/AMLab-Amsterdam/AttentionDeepMIL', 'eb0434ba2795711a45d693d60120ae53532b1b93'),
    ('group_DRO', 'https://github.com/kohpangwei/group_DRO', 'cbbc1c5b06844e46b87e264326b56056d2a437d1'),
    ('HAR-Bench', 'https://github.com/saiketa/HAR-Bench', '358a377929b1b9c0a2cefc417c67f56d15d4d11c'),
    ('IMUEval', 'https://github.com/H-IAAC/synth-imu-eval', '7f9a79187af708b7d7a00ccc947befa059d024f0'),
], columns=['upstream', 'repository', 'audited_commit'])
display(upstream)
upstream.to_csv(PROCESSED / 'prior_method_upstream_commits.csv', index=False)


,upstream,repository,audited_commit
0,InceptionTime,https://github.com/hfawaz/InceptionTime,952d115e83fb3a66d75c5858c702f8dd5eeb18b7
1,MiniROCKET,https://github.com/angus924/minirocket,0b1c245d9c9dbc50886f28bc7b32d5d45b5663d6
2,HAROOD,https://github.com/AIFrontierLab/HAROOD,3c2ce00e2b408ddd913c3d179cd86daf0c44bc90
3,ERM++,https://github.com/piotr-teterwak/erm_plusplus,2535f5d21d16e4fadcec79909835fac97cff6d63
4,AttentionDeepMIL,https://github.com/AMLab-Amsterdam/AttentionDe...,eb0434ba2795711a45d693d60120ae53532b1b93
5,group_DRO,https://github.com/kohpangwei/group_DRO,cbbc1c5b06844e46b87e264326b56056d2a437d1
6,HAR-Bench,https://github.com/saiketa/HAR-Bench,358a377929b1b9c0a2cefc417c67f56d15d4d11c
7,IMUEval,https://github.com/H-IAAC/synth-imu-eval,7f9a79187af708b7d7a00ccc947befa059d024f0


## What the upstream researchers actually implemented

1. **InceptionTime:** six Inception modules, residual shortcuts after every third module, approximately 40/20/10 kernels, 32 filters per branch, ReLU, long training with learning-rate reduction, and a five-network ensemble. The repository model uses two modules, 7/15/25 kernels, 16 filters, GELU, a residual in each module, and short source-selected training. It is a defensible compact CNN but not an InceptionTime reproduction.
2. **MiniROCKET:** the official transform defaults to 10,000 features and 32 maximum dilations, followed by feature scaling and RidgeClassifierCV over logarithmic alpha values. Historical project runs used 2,000 features, 16 dilations, and fixed Ridge alpha 1.0; notebook 31 used logistic regression to obtain probabilities. Those are reduced MiniROCKET variants, not the canonical benchmark.
3. **HAROOD CORAL:** computes each source-domain classification loss and all source-pair mean/covariance penalties. The project reproduces the released mean-plus-covariance penalty and source-pair averaging, changing multiclass cross-entropy to binary BCE appropriately. This is the strongest code match, although the CORAL weight was fixed instead of tuned.
4. **ERM++:** combines pretrained initialization, linear probing, stronger regularization, simple moving average, validation-selected duration, and retraining on all source data. The project implements head-only warm-up, higher weight decay, and SMA on a randomly initialized compact CNN. The name `ERM++-style` is accurate; it is not full ERM++.
5. **GroupDRO:** HAROOD maintains one adversarial weight per source-domain minibatch. The project maintained weights over source×class cells, used one seed, and fixed the duration. The exponential update is correct, but the experimental object is different.
6. **AttentionDeepMIL:** the local `tanh(Vh) × sigmoid(Uh) → softmax` operator and bag-level Bernoulli loss match the official implementation. The adaptation from 'at least one positive instance' bags to a participant diagnosis over gait windows is project-specific and not validated by that paper.
7. **IMUDiffusion:** the paper uses 160-step six-axis sequences, STFT real/imaginary channels, a three-block ResNet/self-attention U-Net, separate sensor schedules, 3,000 denoising steps, 4,500 epochs, Smooth-L1, and one unconditioned generator per activity inside LOSO. Project generators use 200- or 500-point acceleration magnitudes in the time domain, 100–200 steps, about 80–100 epochs, no self-attention, and joint source conditioning. They are valid small DDPM experiments but are not IMUDiffusion reproductions.
8. **IMUEval:** requires complementary fidelity, diversity, discriminative, and predictive tests, including C-FID, JS/MMD, R2R/R2S/S2S DTW, DS and PS. The project realism gates cover spectra, roughness, correlations, nearest neighbours and a discriminator, but not the complete framework.
9. **HAR-Bench normalization:** per-window instance normalization is part of its representation-learning pipeline. The project compared training-fold global z-score and robust scaling, which answer a different question. Its normalization benchmark also repeatedly computed RevalExo results, so those external values must remain descriptive and cannot choose the transform.


In [4]:
alignment = pd.DataFrame([
    ('Compact Inception-style CNN', 'Low', 'High', 'Valid local baseline; not evidence about canonical InceptionTime', 'RETEST canonical implementation'),
    ('HAROOD-style CORAL', 'High core / partial tuning', 'High', 'Standalone and ensemble results are credible for lambda=1 and this backbone', 'KEEP result; tune only in a locked canonical benchmark'),
    ('ERM++-style recipe', 'Partial', 'Moderate', 'Valid local optimization variant; not a reproduction of ERM++', 'KEEP only as locally named ensemble member'),
    ('GroupDRO', 'Partial-low', 'High', 'Does not reject source-domain GroupDRO', 'RETEST only with source groups, multiple seeds, inner tuning'),
    ('MiniROCKET historical/Ridge', 'Partial', 'High', 'Valid reduced transform result; canonical 10k + scaled RidgeCV remains untested', 'RETEST canonical lower-back pipeline'),
    ('MiniROCKET/logistic rescue', 'Partial-low', 'High', 'Rejects only the 2k fixed-logistic fusion', 'Do not generalize to MiniROCKET family'),
    ('Participant attention MIL', 'High operator / low task equivalence', 'Moderate', 'Valid negative for this participant-bag adaptation', 'Do not prioritize without gait-specific bag semantics'),
    ('Healthy DDPM synthesis', 'Low versus IMUDiffusion', 'Moderate input fit', 'Rejects the local small time-domain generators, not IMU diffusion generally', 'Do not rerun until an exact fold-local generator protocol is affordable'),
    ('Physical gain/noise/time augmentation', 'Low versus PPDA', 'Moderate', 'Rejects only the hand-set perturbation ranges', 'Do not call it physics-based synthesis'),
    ('Global/robust normalization', 'Conventional but not HAR-Bench', 'High', 'Internal fold results are valid; RevalExo comparisons are descriptive only', 'Instance-normalization ablation remains untested'),
    ('Binary hard-negative exposure', 'Project-specific', 'High clinical relevance', 'Valid exposure trade-off, not a replication claim', 'Keep as evidence that nonstroke specificity needs paired coverage'),
], columns=['method_family', 'upstream_code_fidelity', 'scenario_fit', 'valid_inference', 'action'])
display(alignment)
alignment.to_csv(PROCESSED / 'prior_method_code_alignment_audit.csv', index=False)
decision = {
    'audit_conclusion': 'Most negative results reject local approximations, not the named method families.',
    'still_valid': ['selected compact lower-back ensemble as an empirical local model', 'HAROOD-style CORAL core result', 'participant/source-disjoint evaluation conclusions', 'specific local augmentation and MIL failures'],
    'claims_to_correct': ['compact CNN is not canonical InceptionTime', 'ERM++-style is not full ERM++', 'GroupDRO source-class run is not source-domain GroupDRO', '2k MiniROCKET variants do not exhaust canonical MiniROCKET', 'local DDPMs are not IMUDiffusion reproductions'],
    'single_next_benchmark': 'Canonical lower-back InceptionTime and canonical 10k MiniROCKET under the notebook-29 three-source, five-seed participant-safe protocol; incumbent compact ensemble included unchanged.',
    'frozen_external_used': False,
}
(PROCESSED / 'prior_method_code_alignment_decision.json').write_text(json.dumps(decision, indent=2), encoding='utf-8')
print(json.dumps(decision, indent=2))


,method_family,upstream_code_fidelity,scenario_fit,valid_inference,action
0,Compact Inception-style CNN,Low,High,Valid local baseline; not evidence about canon...,RETEST canonical implementation
1,HAROOD-style CORAL,High core / partial tuning,High,Standalone and ensemble results are credible f...,KEEP result; tune only in a locked canonical b...
2,ERM++-style recipe,Partial,Moderate,Valid local optimization variant; not a reprod...,KEEP only as locally named ensemble member
3,GroupDRO,Partial-low,High,Does not reject source-domain GroupDRO,"RETEST only with source groups, multiple seeds..."
4,MiniROCKET historical/Ridge,Partial,High,Valid reduced transform result; canonical 10k ...,RETEST canonical lower-back pipeline
5,MiniROCKET/logistic rescue,Partial-low,High,Rejects only the 2k fixed-logistic fusion,Do not generalize to MiniROCKET family
6,Participant attention MIL,High operator / low task equivalence,Moderate,Valid negative for this participant-bag adapta...,Do not prioritize without gait-specific bag se...
7,Healthy DDPM synthesis,Low versus IMUDiffusion,Moderate input fit,Rejects the local small time-domain generators...,Do not rerun until an exact fold-local generat...
8,Physical gain/noise/time augmentation,Low versus PPDA,Moderate,Rejects only the hand-set perturbation ranges,Do not call it physics-based synthesis
9,Global/robust normalization,Conventional but not HAR-Bench,High,Internal fold results are valid; RevalExo comp...,Instance-normalization ablation remains untested


{
  "audit_conclusion": "Most negative results reject local approximations, not the named method families.",
  "still_valid": [
    "selected compact lower-back ensemble as an empirical local model",
    "HAROOD-style CORAL core result",
    "participant/source-disjoint evaluation conclusions",
    "specific local augmentation and MIL failures"
  ],
  "claims_to_correct": [
    "compact CNN is not canonical InceptionTime",
    "ERM++-style is not full ERM++",
    "GroupDRO source-class run is not source-domain GroupDRO",
    "2k MiniROCKET variants do not exhaust canonical MiniROCKET",
    "local DDPMs are not IMUDiffusion reproductions"
  ],
  "single_next_benchmark": "Canonical lower-back InceptionTime and canonical 10k MiniROCKET under the notebook-29 three-source, five-seed participant-safe protocol; incumbent compact ensemble included unchanged.",
  "frozen_external_used": false
}


## Audit decision

The repeated lack of gains does **not** prove that all credible approaches fail. Several experiments used deliberately reduced or scenario-adapted versions and were later interpreted too broadly. The current compact ensemble remains the incumbent because its own source-held-out result is real, not because it reproduces InceptionTime or full ERM++.

There is now one justified corrective benchmark rather than another unrelated idea: compare the untouched incumbent with (a) canonical lower-back InceptionTime mechanics and (b) canonical 10,000-feature MiniROCKET plus scaled RidgeCV, under the exact notebook-29 three-source/five-seed/participant-safe protocol. Calibration must be fitted only inside training sources. No synthetic data, GroupDRO, MIL, threshold search, RevalExo, or NONAN should enter that benchmark.

If neither canonical baseline materially improves both FP and FN without source-level regression, model rotation stops and the project proceeds to new paired-cohort evaluation.
